In [1]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from pydantic import BaseModel,Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_tavily import TavilySearch
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

llm=ChatOpenAI(
    model="qwen3-max",
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
)

d:\develop\miniconda3\envs\llamaindex-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain_tavily'

In [3]:
planner_prompt = ChatPromptTemplate([
    ('system',
     '''你是一名乐于助人的研究助理。根据给定的问题，提出一组需要进行的网页搜索，尽可能全面地回答该问题。请输出5到7个搜索关键词。你必须输出 JSON，并严格按照如下格式：
{{
  "searches": [
    {{
      "query": "搜索关键词",
      "reason": "这样搜索的理由"
    }}
  ]
}}
请严格使用JSONSchema 结构，不要有空格，不要添加额外字段。'''
     ),
    ('human','{query}')
])
class WebSearchItem(BaseModel):
    query: str=Field(description="用于网络搜索的关键词")
    reason: str=Field(description="为什么这个搜索对于解答该问题很重要的理由")
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]=Field(description="为了尽可能全面回答该问题而需要执行的网页搜索列表")
planner_chain=planner_prompt | llm.with_structured_output(WebSearchPlan)
# planner_result = planner_chain.invoke({'query': 'ai在教育应用场景'})
# print(planner_result)

searches=[WebSearchItem(query='AI在教育中的应用场景', reason='直接获取AI在教育领域的主要应用实例和使用场景'), WebSearchItem(query='人工智能教育应用案例', reason='查找具体的AI教育应用成功案例，了解实际落地情况'), WebSearchItem(query='AI个性化学习系统', reason='探索AI如何实现因材施教和个性化教学'), WebSearchItem(query='智能辅导系统 教育', reason='了解AI驱动的辅导工具如何辅助学生学习和教师教学'), WebSearchItem(query='AI教育平台比较', reason='对比不同AI教育平台的功能与适用场景'), WebSearchItem(query='AI在在线教育中的作用', reason='聚焦在线教育环境中AI技术的具体应用与优势'), WebSearchItem(query='教育领域AI技术发展趋势', reason='掌握AI在教育中未来的发展方向和潜在创新点')]


In [4]:
SEARCH_INSTRUCTIONS = (
    '你是一名研究助理。根据提供的搜索词，你需要在网络上进行搜索，并生成一份简明扼要的结果摘要。摘要应包含2到3个段落，总字数不超过300字。务必抓住主要观点，表述简洁，无需使用完整句子或优美语法。这份摘要将供他人用于整合研究报告，因此至关重要的是，你要准确提炼核心内容，忽略任何无关信息。除摘要本身外，不要添加任何额外评论。')
search_tool=TavilySearch(max_results=5,topics='general')
search_agent=create_agent(
    model=llm,
    system_prompt=SEARCH_INSTRUCTIONS,
    tools=[search_tool],
)
# final_message=None
# for step in search_agent.stream(input={'messages': [{'role': 'user', "content": planner_result.searches[0].query}]},stream_mode="values"):
#     step["messages"][-1].pretty_print()
#     final_message=step["messages"][-1]
# search_agent_res=final_message.content
# print('*'*100)
# print(search_agent_res)

================================ Human Message =================================

AI在教育中的应用场景
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_f49ac84c49634d7d83f816cd)
 Call ID: call_f49ac84c49634d7d83f816cd
  Args:
    query: AI在教育中的应用场景
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "AI在教育中的应用场景", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.intel.com.tw/content/www/tw/zh/learn/ai-in-education.html", "title": "人工智慧（AI）在教育界的應用", "content": "學生投入：AI 在教育界的應用可協助語言學習，消弭可能的理解能力差距。學生可透過個人化 AI 支援的應用程式準備考試。此外，他們可以透過同儕、藉由線上討論區的虛擬交流，及課程遊戲化來學習。\n 點名：管理階層和職員可利用 AI 協助掌握學生出席率、記錄已知或無故缺席，甚或利用 AI 通知家長學生遲到。\n 課堂管理：AI 支援系統可協助教師監督學生活動，尤其是個人化學習計畫、小組實驗室、研討會或混合式學習體驗。因此，教師更容易查看學生進度，並有更多時間協助必要的情緒認知學習與行為管理。\n 作文評分：教師可使用 AI 系統先行檢查學生作文的文法、句型和是否抄襲。而且 AI 可協助確定學生是否正確理解作業，及正確遵循寫作提示。\n 學生與課程理解：教師可根據測驗方法，決定考卷是否自動計分。

In [5]:
class ReportData(BaseModel):
    short_summary:str=Field(description="一份2-3句话的简短研究结论摘要")
    markdown_report:str=Field(description="最终生成的报告(markdown格式)")
    follow_up_question:str=Field(description="建议进一步研究的相关主题")
WRITER_PROMPT = (
    "你是一名高级研究员，负责为一个研究问题撰写一份连贯的报告。"
    "你必须使用 **JSON** 格式输出结果，并严格包含以下字段：\n"
    "short_summary, markdown_report, follow_up_question。\n"
    "你将获得原始的研究问题以及由研究助理完成的初步研究内容。"
    "首先，你需要制定一份报告大纲，说明报告的结构和逻辑流程。"
    "接着，生成完整的报告并将其作为最终输出返回。"
    "最终输出应使用Markdown格式，内容应详尽且篇幅较长，目标为10到20页，至少1500字。最终结果请用中文输出"
)
writer_prompt=ChatPromptTemplate([
    ('system',WRITER_PROMPT),
    ('human','{query}')
])
writer_chain=writer_prompt | llm.with_structured_output(ReportData)
# final_query = f'Summarized search results: {search_agent_res}'
# result = writer_chain.invoke({'query': final_query})
# print(result)

short_summary='AI在教育中的应用涵盖个性化学习、智能辅导、自动评估、课堂与行政管理优化，并拓展至美育、劳动教育和心理健康等领域。生成式AI进一步推动互动式与创造性学习，但需警惕数据隐私、算法偏见与伦理风险，确保技术以人为本。' markdown_report='# AI在教育中的应用场景与挑战分析报告\n\n## 引言\n\n随着人工智能（Artificial Intelligence, AI）技术的快速发展，教育领域正经历一场深刻的数字化转型。AI不仅提升了教学效率，还重塑了学习方式与师生关系。本报告系统梳理AI在教育中的主要应用场景，探讨其带来的教育价值，并分析伴随而来的伦理与实践挑战，旨在为教育工作者、政策制定者和技术开发者提供参考。\n\n## 一、AI在教育中的核心应用场景\n\n### 1. 个性化学习\n\nAI通过采集和分析学生的学习行为数据（如答题记录、停留时间、错误模式等），构建学习者画像，进而为其定制个性化的学习路径。例如，自适应学习平台（如Knewton、DreamBox）能动态调整内容难度与顺序，推荐适配的学习资源，帮助学生在“最近发展区”内高效学习。这种以学生为中心的模式，有效弥补了传统“一刀切”教学的不足。\n\n### 2. 智能辅导系统\n\n在语言学习、数学、编程等学科中，AI驱动的智能辅导系统（Intelligent Tutoring Systems, ITS）可提供7×24小时的实时互动支持。例如，Duolingo利用自然语言处理（NLP）技术实现口语与写作反馈；Mathia等平台则通过认知建模模拟人类教师的解题引导策略，帮助学生理解概念而非仅记忆答案。\n\n### 3. 自动化评估与反馈\n\nAI可自动批改选择题、填空题乃至作文与编程作业。基于深度学习的作文评分系统（如ETS的e-rater）能从语法、逻辑、词汇多样性等维度给出评分与修改建议。更重要的是，AI能识别学生的知识薄弱点，生成诊断性报告，帮助教师精准干预。\n\n### 4. 课堂管理与行政事务优化\n\nAI工具可协助教师完成考勤（如人脸识别签到）、课堂行为分析（如注意力监测）、成绩统计、排课等重复性工作。例如，ClassDojo结合AI与行为心理学，鼓励积极课堂行为；学校管理系统集成AI后，可自动生成教学日志、学情周报，大幅减轻教师行政负担

In [6]:
# 生成关键词规划
def plan_searches(query: str) -> WebSearchPlan:
    result = planner_chain.invoke({'query': query})
    print("计划为：",result)
    return result

# 根据关键词进行搜索
def search(item:WebSearchItem) -> str | None:
    try:
        final_query = f"Search Item: {item.query}\nReason for searching: {item.reason}"
        result = search_agent.invoke({"messages":[HumanMessage(final_query)]})
        return str(result['messages'][-1].content)
    except Exception:
        return None

# 根据关键词列表逐个搜索得到搜索结果列表
def perform_searches(search_plan: WebSearchPlan):
    results = []
    for item in search_plan.searches:
        result = search(item)
        if result is not None:
            results.append(result)
    print('摘要列表：',results)
    return results

# 根据搜索的结果列表和用户提问生成报告
def write_report(query: str, search_results) -> ReportData:
    summary=''
    for search_result in search_results:
        summary += search_result
    final_query =  f'Original query: {query}\n Summarized search results: {summary}'
    result=writer_chain.invoke({'query': final_query})
    return result

# 串联以上流程函数
def deepresearch(query: str) -> ReportData:
    '''
    输入一个研究主题，自动完成搜索规划、搜索和写报告
    返回最终的ReportData对象，就是一个markdown的格式完整的研究报告文档
    '''
    search_plan = plan_searches(query)
    print('*'*100)
    search_results = perform_searches(search_plan)
    print('*'*100)
    report = write_report(query, search_results)
    print(report.markdown_report)
deepresearch('ai在教育应用场景')


计划为： searches=[WebSearchItem(query='AI在教育中的应用场景', reason='直接获取AI在教育领域的主要应用案例和场景概述'), WebSearchItem(query='人工智能个性化学习系统', reason='了解AI如何通过个性化教学提升学习效果'), WebSearchItem(query='AI智能辅导与虚拟助教', reason='探索AI在辅导学生和辅助教师方面的具体应用'), WebSearchItem(query='AI自动批改作业与评估系统', reason='研究AI在减轻教师负担、提高评估效率方面的作用'), WebSearchItem(query='AI教育平台案例分析', reason='通过实际平台案例了解AI技术在教育中的落地情况'), WebSearchItem(query='AI在在线教育中的作用', reason='分析AI如何推动在线教育的发展与优化用户体验'), WebSearchItem(query='AI教育应用的挑战与伦理问题', reason='全面了解AI在教育中应用所面临的限制和潜在风险')]
****************************************************************************************************
摘要列表： ['AI在教育中的主要应用场景包括个性化学习、智能辅导、自动化评估与教学管理。通过分析学生的学习数据，AI可定制学习路径、推荐资源并提供实时反馈；同时能自动批改作业、识别知识薄弱点，并辅助教师进行课堂管理和出勤记录。生成式AI还被用于课程设计、教案生成、艺术创作辅助及心理健康教育等多样化教学支持。\n\n此外，AI技术正推动教育公平与特殊教育发展，例如为有阅读障碍或语言障碍的学生转换内容形式，或开发筛查与干预工具。虚拟助教和智能导师系统也逐步实现全天候学习陪伴。然而，应用中仍需关注数据隐私、算法偏见及教育伦理等问题，确保技术符合教学规律并真正服务于育人目标。', '人工智能个性化学习系统通过数据驱动和生成式AI技术，显著提升教学的针对性与效率。系统能实时分析学生的学习行为、知识掌握程度及兴趣偏好，动态调整内容推送与教学策略，实现“千人千面”的学习路径。例如，在英语教学中，